# Multilingual Smart Summarization Colab Notebook

This notebook demonstrates how to run the summarisation pipeline provided in the project without the Streamlit interface. It is designed for Google Colab but can also run in any Jupyter environment with minor adjustments.

**Steps:**

1. Install required packages (Whisper, Transformers, etc.).
2. Mount Google Drive to access media files and save outputs.
3. Clone or upload this project into your drive.
4. Import the modules and run the pipeline on a sample file.


In [ ]:
# Install dependencies
!pip install -q moviepy pydub openai-whisper transformers torch langdetect yake networkx matplotlib sentencepiece
# Install FFmpeg for audio processing
!apt-get update -y && apt-get install ffmpeg -y > /dev/null


In [ ]:
# Mount Google Drive (uncomment if running in Colab)
from google.colab import drive
drive.mount('/content/drive')


### Clone or upload the project

If you have pushed this repository to GitHub, clone it in Colab. Otherwise, upload the project folder to your Drive and adjust the path below.


In [ ]:
# Example: cloning from GitHub (replace <USERNAME> with your GitHub username)
# !git clone https://github.com/<USERNAME>/multilingual-video-audio-summarizer.git /content/multilingual-video-audio-summarizer

# If you uploaded the repository into Google Drive, append that path instead
import sys
project_path = '/content/drive/MyDrive/multilingual-video-audio-summarizer'  # adjust if needed
sys.path.append(project_path)


### Run the pipeline on a sample file

Place a sample media file (audio or video) in your Google Drive, update `input_path` accordingly and run the following cell. The outputs will be stored in your Drive.


In [ ]:
from modules import (
    extract_audio,
    transcribe_audio,
    clean_text,
    detect_language,
    hierarchical_summarize,
    extract_keywords,
    build_keyword_graph,
    save_transcript,
    save_clean_transcript,
    save_chunk_summaries,
    save_final_summary,
    save_keywords,
)
from modules.config import OUTPUT_DIR
import os

# Path to your input media file on Google Drive
input_path = '/content/drive/MyDrive/sample_data/sample_audio.mp3'  # change this

# Ensure output directory exists on Drive
output_dir = '/content/drive/MyDrive/summarizer_outputs'
os.makedirs(output_dir, exist_ok=True)

# Run pipeline
print('Extracting audio...')
audio_path = extract_audio(input_path)
print('Transcribing audio...')
transcript, whisper_lang = transcribe_audio(audio_path)

cleaned = clean_text(transcript)
lang = detect_language(cleaned) or whisper_lang or 'en'
print(f'Detected language: {lang}')

final_summary, chunk_summaries = hierarchical_summarize(cleaned, lang)
keywords = extract_keywords(cleaned, lang)

# Build keyword graph and save to Drive
graph_path = build_keyword_graph(keywords, cleaned, os.path.join(output_dir, 'keyword_graph.png'))

# Save artefacts
base = os.path.splitext(os.path.basename(input_path))[0]
save_transcript(transcript, os.path.join(output_dir, f'{base}_transcript.txt'))
save_clean_transcript(cleaned, os.path.join(output_dir, f'{base}_cleaned.txt'))
save_chunk_summaries(chunk_summaries, os.path.join(output_dir, f'{base}_chunks.txt'))
save_final_summary(final_summary, os.path.join(output_dir, f'{base}_summary.txt'))
save_keywords(keywords, os.path.join(output_dir, f'{base}_keywords.txt'))

print('Processing complete.')
print('Outputs saved to', output_dir)
